# Baseline smoke test (BD-01 to BD-04)

**Outcome:** The historical arraignment decision to set bail or remand.
It is not actual detention or a measure of future behavior.

**Design:** NYC main sample. Train and tune on 2021–2022; evaluate on 2023.
The notebook builds the development data in memory from the team's pinned
source files and does not read the 2024 source file, write sample-flow reports,
or evaluate the final test. It runs a charge-based rule and two L2 logistic
models: CASE and the prespecified CASE_PLUS_GENDER diagnostic.

Open from the repository's notebooks directory in the Python 3.11 pol408
environment. Aggregate output is shown below; case predictions and fitted
artifacts are not part of the Git submission.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
from importlib.metadata import version

import pandas as pd

sys.path.append("..")
from src import config, features
from src.evaluate import metrics
from src.models import BASELINE_MODELS, fit_and_predict, split_by_year
from src.rules import rule_scores
from src.sample import build_sample, load_raw

assert config.ROOT == Path.cwd().resolve().parent, "Run from the repository notebooks directory."
print(f"Python: {sys.version.split()[0]}")
print({p: version(p) for p in ["pandas", "numpy", "scikit-learn", "pyarrow"]})

Python: 3.11.18
{'pandas': '3.1.1', 'numpy': '2.4.6', 'scikit-learn': '1.9.1', 'pyarrow': '26.0.1'}


In [2]:
development_years = [*config.TRAIN_YEARS, config.VALID_YEAR]
assert config.TEST_YEAR not in development_years
raw = load_raw(years=development_years)
assert set(raw[config.YEAR_COL].dropna().unique()) == set(development_years)
development, flow = build_sample(raw)
del raw

published = pd.read_csv(config.SAMPLE_FLOW_FILE).set_index("step").loc["MAIN"]
expected_counts = {year: int(published[str(year)]) for year in development_years}
columns = [
    config.ID_COL, config.YEAR_COL, config.SPLIT_COL, config.MAIN_SAMPLE_COL,
    config.TARGET_Q1, *features.ALL_SETS["CASE_PLUS_GENDER"],
]
main = development.loc[development[config.MAIN_SAMPLE_COL], columns].copy()
counts = main.groupby(config.YEAR_COL).size().to_dict()
assert counts == expected_counts, (counts, expected_counts)
assert set(main[config.SPLIT_COL]) == {"train", "valid"}
print("Main cases by year:", counts)
print("Cases by split:", {k: len(v) for k, v in split_by_year(main).items() if k != "test"})
del development

  read NYS for Web 2021.csv.gz: 246,202 rows (124,264 NYC)


  read NYS for Web 2022.csv.gz: 286,588 rows (150,585 NYC)


  read NYS for Web 2023.csv.gz: 324,619 rows (175,658 NYC)


Main cases by year: {np.int64(2021): 89189, np.int64(2022): 104007, np.int64(2023): 120457}


Cases by split: {'train': 193196, 'valid': 120457}


The rule estimates historical outcome rates by charge grade and
violent felony status. Small cells back off to their charge grade; unseen
grades use the training base rate. Training rule scores are fold-fitted.

For each logistic model, preprocessing and tuning occur within training-year
cross-validation. After tuning, the model fitted on both training years
scores 2023. Training OOF scores reuse parameters selected on these same
training data, so the 2023 score is the main comparison.

In [3]:
predictions = [rule_scores(main)]
model, grid = BASELINE_MODELS["logit"]
for feature_set in ["CASE", "CASE_PLUS_GENDER"]:
    print("Fitting", feature_set, flush=True)
    scores = fit_and_predict(
        main, "logit", model, grid, feature_set, run_test=False,
    )
    print("Selected parameters:", scores.attrs["best_params"], flush=True)
    predictions.append(scores)
assert all(set(pred["split"]) == {"train_oof", "valid"} for pred in predictions)

Fitting CASE


Selected parameters: {'clf__C': 1}


Fitting CASE_PLUS_GENDER


Selected parameters: {'clf__C': 10}


In [4]:
summary = pd.DataFrame([
    {"model": p["model"].iloc[0], "feature_set": p["feature_set"].iloc[0],
     "split": split, **metrics(part["y_true"], part["y_prob"])}
    for p in predictions
    for split, part in p.groupby("split", sort=False)
])
print(summary.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

base, plus = [
    p.loc[p["split"].eq("valid")].set_index(config.ID_COL).sort_index()
    for p in predictions[1:]
]
assert base.index.equals(plus.index)
assert base["y_true"].equals(plus["y_true"])
auc_case = metrics(base["y_true"], base["y_prob"])["roc_auc"]
auc_plus = metrics(plus["y_true"], plus["y_prob"])["roc_auc"]
print(f"2023 paired-sample AUC difference (plus gender minus CASE): {auc_plus - auc_case:+.6f}")
if summary.loc[summary["split"].eq("valid"), "roc_auc"].gt(0.95).any():
    print("Review outcome timing and potential leakage before interpreting high AUC.")
print("2024 source file not read or scored.")

model      feature_set     split  roc_auc  avg_precision    brier      n  base_rate
 rule             CASE train_oof 0.880943       0.580378 0.072087 193196   0.129294
 rule             CASE     valid 0.846492       0.478021 0.080927 120457   0.128179
logit             CASE train_oof 0.936537       0.716173 0.060774 193196   0.129294
logit             CASE     valid 0.920289       0.662465 0.067290 120457   0.128179
logit CASE_PLUS_GENDER train_oof 0.937980       0.722509 0.060000 193196   0.129294
logit CASE_PLUS_GENDER     valid 0.921347       0.667988 0.066639 120457   0.128179


2023 paired-sample AUC difference (plus gender minus CASE): +0.001058
2024 source file not read or scored.


The supplementary comparison measures incremental predictive signal
for historical decisions on the same validation dockets. It does not
identify discriminatory treatment or establish that case information fully
accounts for group disparities. Prior-record and pending-case reference
dates remain a documented source limitation (data/README.md, D14).